In [1]:
import json
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import matplotlib as mpl
import matplotlib.pyplot as plt
from IPython.display import Image
%matplotlib inline

import requests
import zipfile
import os.path

In [2]:
cd /home/bhueth/repos/rurec

/home/bhueth/repos/rurec


# Download Data: NAICS, FIPS codes, COA, CBP
- [Census, North American Classification System (NAICS)](https://www.census.gov/eos/www/naics/downloadables/downloadables.html)

- [Census, American National Standards Institute (ANSI) Codes]('https://www.census.gov/geo/reference/ansi.html')

- [USDA Census of Agriculture](ftp://ftp.nass.usda.gov/)

- [Census Cartographic Boundary Shapefiles](https://www.census.gov/geo/maps-data/data/tiger-cart-boundary.html)

- [InfoGroup Business Data (via UW Extension](http://bdrc.uwex.edu/our-databases.iegc)

In [3]:
excluded_states = {'AK', 'PR', 'HI', 'VI', 'DC'}

file_names = ['2012_NAICS_Index_File.xls', 
              '2-digit_2012_Codes.xls',
              'state_fips_master.csv',
              'cbp12st.zip']


urls = ['https://www.census.gov/eos/www/naics/2012NAICS/',
        'https://www.census.gov/eos/www/naics/2012NAICS/',
        'https://raw.githubusercontent.com/kjhealy/fips-codes/master/',
        'https://www2.census.gov/programs-surveys/cbp/datasets/2012/']

for fi, url in zip(file_names, urls):
    if not os.path.isfile('data/' + fi):
        with open('data/' + fi, 'wb') as f:
            r = requests.get(url + fi)
            f.write(r.content)
            if fi.split('.')[-1] == 'zip':
                z = zipfile.ZipFile('data/' + fi, 'r')
                z.extractall('data/')      

if not os.path.isfile('data/qs.census2012.txt.gz'):
    !cd data
    !wget 'ftp://ftp.nass.usda.gov/quickstats/qs.census2012.txt.gz'
    !tar -xzf 'qs.census2012.txt.gz'

areas = ['csa', 'cbsa']
for area in areas:
    if not os.path.isfile('data/' + area + '.zip'): 
        f = 'cb_2017_us_' + area + '_500k.zip'
        url = 'https://www2.census.gov/geo/tiger/GENZ2017/shp/' + f
        r = requests.get(url)
        with open('data/' + area + '.zip', 'wb') as f:
            f.write(r.content)
        z = zipfile.ZipFile('data/' + area + '.zip', 'r')
        z.extractall('data/' + area)


# Raw files (obtained throu UWEX/BDRC) stored at Google Big Query        
if not os.path.isfile('data/ig_2015.gz'):
    query = '''
    SELECT
      state,
      substr(naics, 1, 6) as naics,
      employees,
      latitude,
      longitude
    FROM
      `original.data`
    where year = 2015;
    '''
    df = pd.read_gbq(query, dialect='standard', project_id='info-group-162919')
    df.to_pickle('data/ig_2015.gz')

    df = pd.read_pickle('data/ig_2015.gz')
    df = df[~df.state.isin(excluded_states)]
    df = df.drop('state', axis=1)
    df.employees = pd.to_numeric(df.employees)
    df = df.dropna(subset=['employees'])

# Read NAICS codes and generate groups

In [5]:
file_name = '2-digit_2012_Codes.xls'
df_4 = pd.read_excel('data/' + file_name, 
                     names=['no', 'naics', 'description'],
                     dtypes={'naics': str, 'description': str}).drop(columns='no')
 

# 'farming': ['111', '112', '113', '114']
naics_sectors = {'ag_services': ['115'], 'mining_oil': ['21'],'utilities': ['22'],
         'construction': ['23'], 'manufacturing': ['3'], 'wholesale': ['42'],
         'retail': ['44', '45'], 'transport_storage': ['48', '49'],
         'information': ['51'],  'finance_ins': ['52'],
         'real_estate_lease': ['53'], 'prof_sci_tech': ['54'],
         'mgmt_co': ['55'], 'admin_waste_remed': ['56'],
         'ed_service': ['61'], 'health_social': ['62'],
         'art_rec_enter': ['71'], 'accomm_food_service': ['72'],
         'other_service': ['81'], 'public_admin': ['92']
        }

# six-digit codes
file_name = '2012_NAICS_Index_File.xls'
df = pd.read_excel('data/' + file_name, names=['naics', 'description'],
                   dtypes={'naics': str, 'description': str})


df.description = [d.lower() for d in df.description]
df.naics = df.naics.astype(str)


df_4.naics = df_4.naics.astype('str', inplace=True)
df_4 = df_4[df_4.naics.str.len() == 4]

for n, r in df_4.iterrows():
    df.loc[df.naics.str.slice(0,4) == r.naics, 'naics4'] = r.naics
    df.loc[df.naics.str.slice(0,4) == r.naics, 'naics4_desc'] = r.description


naics_fai = pd.DataFrame()
for fai_sector in ['food', 'agriculture']:
    
    str_match = 'food'
    if fai_sector == 'agriculture':
        str_match = 'farm | agri | animal'
    
    for sector, sec_codes in naics_sectors.items():
        
        # 'retail' and 'transport_storage' sectors contain multiple 2-digit sectors codes
        tmp_str = ' | '.join("df.naics.str.startswith('{}')".format(s) 
                             for s in sec_codes)

        # get all industries in sector
        tmp = df[eval(tmp_str)]
        
        # get subset of industries from sector in fai_code
        tmp = tmp[tmp.description.str.contains(str_match)]
        tmp['sector'] = len(tmp) * [sector]
        tmp['fai_sector'] = len(tmp) * [fai_sector]
        
        if len(tmp) > 0:
            naics_fai = naics_fai.append(tmp)

# OJO: Need to go through codes by hand
non_fai_naics = ['561710'] # pest control excpet aricultural ...
naics_fai = naics_fai[~naics_fai.naics.isin(non_fai_naics)]
naics_fai[['naics4', 'naics4_desc', 'naics', 'description']].set_index(['naics4', 'naics']).sort_index()


naics4_desc  \
naics4 naics                                                       
1151   115114             Support Activities for Crop Production   
       115115             Support Activities for Crop Production   
       115115             Support Activities for Crop Production   
       115116             Support Activities for Crop Production   
1152   115210           Support Activities for Animal Production   
       115210           Support Activities for Animal Production   
2362   236210               Nonresidential Building Construction   
       236220               Nonresidential Building Construction   
2371   237120                        Utility System Construction   
2379   237990     Other Heavy and Civil Engineering Construction   
2382   238290                     Building Equipment Contractors   
3111   311111                          Animal Food Manufacturing   
       311111                          Animal Food Manufacturing   
       311111                          Animal Food Manufacturing   
       311111                          Animal Food Manufacturing   
       311111                          Animal Food Manufacturing   
       311111                          Animal Food Manufacturing   
       311111                          Animal Food Manufacturing   
       311119                          Animal Food Manufacturing   
       311119                          Animal Food Manufacturing   
       311119                          Animal Food Manufacturing   
       311119                          Animal Food Manufacturing   
       311119                          Animal Food Manufacturing   
       311119                          Animal Food Manufacturing   
       311119                          Animal Food Manufacturing   
       311119                          Animal Food Manufacturing   
       311119                          Animal Food Manufacturing   
       311119                          Animal Food Manufacturing   
       311119                          Animal Food Manufacturing   
       311119                          Animal Food Manufacturing   
...                                                          ...   
5417   541711       Scientific Research and Development Services   
       541712       Scientific Research and Development Services   
5419   541940  Other Professional, Scientific, and Technical ...   
       541940  Other Professional, Scientific, and Technical ...   
       541940  Other Professional, Scientific, and Technical ...   
6242   624210  Community Food and Housing, and Emergency and ...   
7121   712130  Museums, Historical Sites, and Similar Institu...   
       712130  Museums, Historical Sites, and Similar Institu...   
7223   722310                              Special Food Services   
       722310                              Special Food Services   
       722310                              Special Food Services   
       722310                              Special Food Services   
       722310                              Special Food Services   
       722310                              Special Food Services   
       722310                              Special Food Services   
       722330                              Special Food Services   
       722330                              Special Food Services   
       722330                              Special Food Services   
       722330                              Special Food Services   
7225   722513                Restaurants and Other Eating Places   
       722513                Restaurants and Other Eating Places   
8113   811310  Commercial and Industrial Machinery and Equipm...   
       811310  Commercial and Industrial Machinery and Equipm...   
       811310  Commercial and Industrial Machinery and Equipm...   
8129   812910                           Other Personal Services    
8134   813410                    Civic and Social Organizations    
8139   813910  Business, Professional, La